In [9]:
# Importing necessary libraries
import numpy as np
import joblib
from tensorflow.keras.models import load_model
from scipy.optimize import minimize

import warnings
warnings.filterwarnings("ignore")


In [10]:
# Load the necessary artifacts
model = load_model('../artifacts/model.h5')
scaler = joblib.load('../artifacts/scaler.pkl')

# Define objective function
def objective(inputs):
    inputs_scaled = scaler.transform([inputs])
    predicted = model.predict(inputs_scaled, verbose=0)[0][0]
    return predicted  # minimizes σ/σmax

# Bounds: [1/T, ln_strain_rate, Strain]
bounds = [
    (0.000755, 0.000817),  # 1/T range
    (-2.302585, 2.708050), # ln strain rate range
    (0.1, 0.7)             # strain range
]

# Initial guess — midpoint of each range
x0 = [0.000786, 0.2, 0.4]

# Run optimization
result = minimize(
    objective,
    x0,
    method='L-BFGS-B',
    bounds=bounds
)

print("Optimal Inputs Found:")
print(f"  1/T            : {result.x[0]:.6f}")
print(f"  ln strain rate : {result.x[1]:.6f}")
print(f"  Strain         : {result.x[2]:.6f}")
print(f"  Predicted σ/σmax: {result.fun:.4f}")

Optimal Inputs Found:
  1/T            : 0.000755
  ln strain rate : 0.200000
  Strain         : 0.400000
  Predicted σ/σmax: 0.5341


In [11]:
# Conversion to engineering units
optimal_inv_T = result.x[0]
optimal_ln_sr = result.x[1]
optimal_strain = result.x[2]

# Convert back to engineering units
optimal_T_kelvin = 1 / optimal_inv_T
optimal_T_celsius = optimal_T_kelvin - 273.15
optimal_strain_rate = np.exp(optimal_ln_sr)

print("=" * 45)
print("  OPTIMAL PROCESS PARAMETERS — AISI 304")
print("=" * 45)
print(f"  Temperature   : {optimal_T_celsius:.1f} °C")
print(f"  Strain Rate   : {optimal_strain_rate:.4f} s⁻¹")
print(f"  Strain        : {optimal_strain:.2f}")
print(f"  Min σ/σmax    : {result.fun:.4f}")
print("=" * 45)

  OPTIMAL PROCESS PARAMETERS — AISI 304
  Temperature   : 1051.4 °C
  Strain Rate   : 1.2214 s⁻¹
  Strain        : 0.40
  Min σ/σmax    : 0.5341


## Optimization Conclusions

The trained MLP was used as a differentiable surrogate model to
determine the optimal hot deformation parameters for AISI 304
stainless steel. Optimization was performed using the L-BFGS-B
gradient-based algorithm via scipy.optimize.minimize, which
navigates the smooth ANN prediction surface to find the input
combination that minimizes normalized flow stress.

**Why Minimize Flow Stress**
In hot deformation processing, minimizing flow stress is the
primary engineering objective. Lower flow stress corresponds to
conditions where the material offers least resistance to
deformation, reducing the energy and force requirements of the
forming process, minimizing tool wear, and improving the overall
efficiency of hot working operations.

**Why ANN as Surrogate**
Tree-based models such as XGBoost and Random Forest produce
discontinuous, step-like prediction surfaces with no mathematical
gradient. Gradient-based optimization requires a smooth,
differentiable function. The ANN, built entirely from differentiable
operations, uniquely satisfies this requirement, making it the
only model among those evaluated that supports this optimization
approach.

**Optimization Bounds**
The search space was constrained to the physical range of the
experimental data:

| Parameter     | Lower Bound      | Upper Bound      |
|---------------|------------------|------------------|
| Temperature   | 0.000755 K⁻¹     | 0.000817 K⁻¹     |
| ln Strain Rate| -2.3026          | 2.7081           |
| Strain        | 0.1              | 0.7              |

**Optimal Parameters Found**

| Parameter     | Optimal Value    |
|---------------|------------------|
| Temperature   | 1051.4 °C        |
| Strain Rate   | 1.2214 s⁻¹       |
| Strain        | 0.40             |
| Min σ/σmax    | 0.5341           |

**Interpretation**
The optimization converged to a deformation temperature of 1051.4°C,
a strain rate of 1.2214 s⁻¹, and a strain of 0.40. At these
conditions the model predicts a normalized flow stress of 0.5341,
the lowest achievable value within the physical bounds of the
experimental dataset. These represent the most energy-efficient
hot working parameters for AISI 304 stainless steel within the
tested operating range.

The result is physically consistent, higher temperature reduces
flow stress by increasing atomic mobility and promoting dynamic
recovery, while the intermediate strain rate avoids the stress
hardening associated with very high strain rates.